In [3]:
import pandas as pd


column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race',
    'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]


df = pd.read_csv(
    "adult.csv",
    header=None,
    names=column_names,
    na_values='?',
    skipinitialspace=True
)


print(" Erste 5 Zeilen:\n")
print(df.head(), "\n")


print(" Anzahl fehlender Werte pro Spalte:\n")
print(df.isnull().sum(), "\n")


cols = ['age', 'education-num', 'hours-per-week']

print("Statistische Kennzahlen:\n")
for col in cols:
    mean = df[col].mean()
    median = df[col].median()
    mode = df[col].mode()[0]
    var = df[col].var()
    std = df[col].std()
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    print(f"Kennzahlen für '{col}':")
    print(f"  - Arithmetisches Mittel: {mean:.2f}")
    print(f"  - Median: {median}")
    print(f"  - Modus: {mode}")
    print(f"  - Varianz: {var:.2f}")
    print(f"  - Standardabweichung: {std:.2f}")
    print(f"  - Interquartilsabstand (IQR): {iqr:.2f}")
    print("-" * 60)


 Erste 5 Zeilen:

   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital-status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0         

Schritt 4; Aufgabe 1


In [13]:
import pandas as pd
import numpy as np

column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race',
    'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]

df = pd.read_csv(
    "adult.csv",
    header=None,
    names=column_names,
    na_values=' ?',
    skipinitialspace=True
)

for col in ['age', 'education-num', 'hours-per-week']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['age', 'hours-per-week'])


def categorize_age(age):
    if pd.isna(age):
        return np.nan
    age = int(age)
    if 18 <= age <= 30:
        return "jung (18–30)"
    elif 31 <= age <= 50:
        return "mittel (31–50)"
    else:
        return "alt (>50)"


df["age_group"] = df["age"].apply(categorize_age)

order = ["jung (18–30)", "mittel (31–50)", "alt (>50)"]
df["age_group"] = pd.Categorical(df["age_group"], categories=order, ordered=True)
df = df[df["age_group"].notna()]

agg_df = df.groupby("age_group", observed=True)[["hours-per-week"]].agg({"hours-per-week": ["median", "var"]})
agg_df.columns = ["median", "var"]
stats = agg_df.reset_index().sort_values("age_group")
stats["median"] = stats["median"].round(2)
stats["var"] = stats["var"].round(2)

print("Median und Varianz der wöchentlichen Arbeitsstunden je Altersgruppe:\n")
print(stats.to_string(index=False))
print()

print("Interpretation:")
medians = stats["median"]
if medians.nunique() == 1:
    print(f"- Typische Arbeitszeit (Median): Alle Altersgruppen haben denselben Median ({medians.iloc[0]} Stunden).")
else:
    max_med = medians.max()
    groups_with_max_med = stats.loc[stats["median"] == max_med, "age_group"].tolist()
    print(
        f"- Altersgruppe(n) mit typischerweise den meisten Arbeitsstunden (höchster Median {max_med}): {', '.join(groups_with_max_med)}")

max_var = stats["var"].max()
groups_with_max_var = stats.loc[stats["var"] == max_var, "age_group"].tolist()
if len(groups_with_max_var) == 1:
    print(f"- Größte Variabilität der Arbeitszeit (höchste Varianz {max_var}): {groups_with_max_var[0]}")
else:
    print(f"- Größte Variabilität der Arbeitszeit (höchste Varianz {max_var}): {', '.join(groups_with_max_var)}")

print("\nZusammenfassung der Arbeitsstunden je Altersgruppe:\n")
print(df.groupby("age_group", observed=True)["hours-per-week"].describe().round(2))


Median und Varianz der wöchentlichen Arbeitsstunden je Altersgruppe:

     age_group  median    var
  jung (18–30)    40.0 147.31
mittel (31–50)    40.0 118.30
     alt (>50)    40.0 201.43

Interpretation:
- Typische Arbeitszeit (Median): Alle Altersgruppen haben denselben Median (40.0 Stunden).
- Größte Variabilität der Arbeitszeit (höchste Varianz 201.43): alt (>50)

Zusammenfassung der Arbeitsstunden je Altersgruppe:

                  count   mean    std  min   25%   50%   75%   max
age_group                                                         
jung (18–30)    10177.0  37.67  12.14  1.0  32.0  40.0  40.0  99.0
mittel (31–50)  15529.0  43.32  10.88  1.0  40.0  40.0  50.0  99.0
alt (>50)        6855.0  38.01  14.19  1.0  35.0  40.0  42.0  99.0


Schritt 4; Aufgabe 2

In [6]:
import pandas as pd

column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race',
    'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]

df = pd.read_csv(
    "adult.csv",
    header=None,
    names=column_names,
    na_values=' ?',
    skipinitialspace=True
)

# Schritt 1: Zwei Einkommensgruppen bilden
income_groups = df.groupby("income")["education-num"].agg(["mean", "median"]).round(2).reset_index()

print("Arithmetisches Mittel und Median der Bildungsjahre je Einkommensgruppe:\n")
print(income_groups)

# Schritt 2: Interpretation vorbereiten
mean_low = income_groups.loc[income_groups["income"] == "<=50K", "mean"].values[0]
mean_high = income_groups.loc[income_groups["income"] == ">50K", "mean"].values[0]

median_low = income_groups.loc[income_groups["income"] == "<=50K", "median"].values[0]
median_high = income_groups.loc[income_groups["income"] == ">50K", "median"].values[0]

print("\nInterpretation:")
if mean_high > mean_low:
    print(
        f"Personen mit höherem Einkommen (>50K) haben im Durchschnitt ({mean_high}) mehr Bildungsjahre als Personen mit niedrigerem Einkommen ({mean_low}).")
if median_high > median_low:
    print(
        f"Auch der Median des Bildungsniveaus ist höher ({median_high} vs. {median_low}), was zeigt, dass das höhere Bildungsniveau allgemein verbreitet ist.")


Arithmetisches Mittel und Median der Bildungsjahre je Einkommensgruppe:

  income   mean  median
0  <=50K   9.60     9.0
1   >50K  11.61    12.0

Interpretation:
Personen mit höherem Einkommen (>50K) haben im Durchschnitt (11.61) mehr Bildungsjahre als Personen mit niedrigerem Einkommen (9.6).
Auch der Median des Bildungsniveaus ist höher (12.0 vs. 9.0), was zeigt, dass das höhere Bildungsniveau allgemein verbreitet ist.


Schritt 4; Aufgabe 3

In [7]:
import pandas as pd

# Spaltennamen definieren
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race',
    'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]

# Datensatz laden
df = pd.read_csv(
    "adult.csv",
    header=None,
    names=column_names,
    na_values=' ?',
    skipinitialspace=True
)

# Schritt 3: Arithmetisches Mittel und Standardabweichung der Arbeitsstunden nach Geschlecht
gender_stats = (
    df.groupby("sex")["hours-per-week"]
    .agg(["mean", "std"])
    .round(2)
    .reset_index()
)

print("Arithmetisches Mittel und Standardabweichung der Wochenarbeitszeit nach Geschlecht:\n")
print(gender_stats)

# Interpretation
mean_male = gender_stats.loc[gender_stats["sex"] == "Male", "mean"].values[0]
mean_female = gender_stats.loc[gender_stats["sex"] == "Female", "mean"].values[0]

std_male = gender_stats.loc[gender_stats["sex"] == "Male", "std"].values[0]
std_female = gender_stats.loc[gender_stats["sex"] == "Female", "std"].values[0]

print("\nInterpretation:")
if mean_male > mean_female:
    print(f"Männer arbeiten im Durchschnitt länger pro Woche ({mean_male} Stunden) als Frauen ({mean_female} Stunden).")
else:
    print(f"Frauen arbeiten im Durchschnitt länger pro Woche ({mean_female} Stunden) als Männer ({mean_male} Stunden).")

if std_male > std_female:
    print(
        f"Die Streuung der Arbeitszeit ist bei Männern größer ({std_male}) als bei Frauen ({std_female}), was auf unterschiedlichere Arbeitszeitmodelle hindeutet.")
else:
    print(f"Die Streuung der Arbeitszeit ist bei Frauen größer ({std_female}) als bei Männern ({std_male}).")


Arithmetisches Mittel und Standardabweichung der Wochenarbeitszeit nach Geschlecht:

      sex   mean    std
0  Female  36.41  11.81
1    Male  42.43  12.12

Interpretation:
Männer arbeiten im Durchschnitt länger pro Woche (42.43 Stunden) als Frauen (36.41 Stunden).
Die Streuung der Arbeitszeit ist bei Männern größer (12.12) als bei Frauen (11.81), was auf unterschiedlichere Arbeitszeitmodelle hindeutet.


Schritt 4; Aufgabe 4

In [14]:
import pandas as pd

column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race',
    'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]

df = pd.read_csv(
    "adult.csv",
    header=None,
    names=column_names,
    na_values=' ?',
    skipinitialspace=True
)

high_income = df[df["income"] == ">50K"]
modus_edu = high_income["education"].mode()[0]

print("Häufigster Bildungsabschluss (Modus) bei Personen mit hohem Einkommen (>50K):\n")
print(modus_edu)

print("\nInterpretation:")
print(f"- Der häufigste Bildungsabschluss unter Personen mit Einkommen >50K ist '{modus_edu}'.")
print("- Das weist darauf hin, dass höhere Einkommen meist mit einem höheren Bildungsniveau zusammenhängen,")
print("  da Personen mit diesem Abschluss typischerweise qualifiziertere Berufe und bessere Gehaltschancen haben.")


Häufigster Bildungsabschluss (Modus) bei Personen mit hohem Einkommen (>50K):

Bachelors

Interpretation:
- Der häufigste Bildungsabschluss unter Personen mit Einkommen >50K ist 'Bachelors'.
- Das weist darauf hin, dass höhere Einkommen meist mit einem höheren Bildungsniveau zusammenhängen,
  da Personen mit diesem Abschluss typischerweise qualifiziertere Berufe und bessere Gehaltschancen haben.


Schritt 4; Aufgabe 5

In [16]:
import pandas as pd

column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race',
    'sex', 'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]

df = pd.read_csv(
    "adult.csv",
    header=None,
    names=column_names,
    na_values=' ?',
    skipinitialspace=True
)

quantiles = (
    df.groupby("income")["age"]
    .quantile([0.25, 0.75])
    .unstack()
    .rename(columns={0.25: "25%-Quantil", 0.75: "75%-Quantil"})
    .round(2)
    .reset_index()
)

print("25%- und 75%-Quantil des Alters je Einkommensgruppe:\n")
print(quantiles.to_string(index=False))

age_low_25 = quantiles.loc[quantiles["income"] == "<=50K", "25%-Quantil"].values[0]
age_high_25 = quantiles.loc[quantiles["income"] == ">50K", "25%-Quantil"].values[0]
age_low_75 = quantiles.loc[quantiles["income"] == "<=50K", "75%-Quantil"].values[0]
age_high_75 = quantiles.loc[quantiles["income"] == ">50K", "75%-Quantil"].values[0]

print("\nInterpretation:")
print(f"- Personen mit Einkommen <=50K liegen zu 50% im Altersbereich von etwa {age_low_25} bis {age_low_75} Jahren.")
print(f"- Personen mit Einkommen >50K liegen zu 50% im Altersbereich von etwa {age_high_25} bis {age_high_75} Jahren.")
print("- Daraus wird ersichtlich, dass Personen mit höherem Einkommen im Durchschnitt älter sind,")
print("  da sie typischerweise mehr Berufserfahrung und Zeit für Karriereentwicklung haben.")


25%- und 75%-Quantil des Alters je Einkommensgruppe:

income  25%-Quantil  75%-Quantil
 <=50K         25.0         46.0
  >50K         36.0         51.0

Interpretation:
- Personen mit Einkommen <=50K liegen zu 50% im Altersbereich von etwa 25.0 bis 46.0 Jahren.
- Personen mit Einkommen >50K liegen zu 50% im Altersbereich von etwa 36.0 bis 51.0 Jahren.
- Daraus wird ersichtlich, dass Personen mit höherem Einkommen im Durchschnitt älter sind,
  da sie typischerweise mehr Berufserfahrung und Zeit für Karriereentwicklung haben.
